# Ventura Flow — Colab Setup
Run cells top-to-bottom on first launch.

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/VenturaFlow'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print('Files:', os.listdir('.'))

In [ ]:
# Cell 2: Install dependencies (takes ~3 min on first run)
!pip install -q vllm transformers accelerate huggingface-hub openai python-dotenv

In [ ]:
# Cell 3: Verify GPU
!nvidia-smi
import torch
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\nAvailable VRAM: {vram:.1f} GB')
print('Recommended quant:', '8-bit (bitsandbytes)' if vram >= 38 else '4-bit (bitsandbytes)')

In [ ]:
# Cell 4: Hugging Face login (needed to download DeepSeek weights)
from huggingface_hub import login
login()  # paste your HF token when prompted -- get it from huggingface.co/settings/tokens

In [ ]:
# Cell 5: Launch vLLM server (runs in background, exposes port 8000)
# First run will download ~65GB of weights -- takes 10-15 min
import subprocess, torch, os

vram = torch.cuda.get_device_properties(0).total_memory / 1e9
MODEL_ID = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-32B'
API_KEY  = 'local-dev-key'

cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_ID,
    '--host', '0.0.0.0',
    '--port', '8000',
    '--dtype', 'float16',
    '--max-model-len', '8192',
    '--gpu-memory-utilization', '0.90',
    '--served-model-name', 'deepseek-r1-32b',
    '--trust-remote-code',
    '--api-key', API_KEY,
    '--quantization', 'bitsandbytes',
    '--load-format', 'bitsandbytes',
]

print('Starting server... (this will download weights on first run)')
log_file = open('/content/vllm_server.log', 'w')
proc = subprocess.Popen(cmd, stdout=log_file, stderr=log_file)
print(f'Server PID: {proc.pid} -- tailing logs below:')
!tail -f /content/vllm_server.log

In [ ]:
# Cell 6: Expose port 8000 via ngrok so your local Mac can reach it
# Run this AFTER the server prints 'Application startup complete'
!pip install -q pyngrok
from pyngrok import ngrok

# Get a free token at dashboard.ngrok.com
NGROK_TOKEN = ''  # paste your ngrok authtoken here
ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
print(f'\n=== PASTE THIS INTO bull_agent.py ===')
print(f'BASE_URL = "{public_url}/v1"')

In [ ]:
# Cell 7: Quick smoke test from inside Colab
from openai import OpenAI
client = OpenAI(base_url='http://localhost:8000/v1', api_key='local-dev-key')
r = client.chat.completions.create(
    model='deepseek-r1-32b',
    messages=[{'role': 'user', 'content': 'Reply with: SERVER OK'}],
    max_tokens=10,
)
print(r.choices[0].message.content)